In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [3]:
dataset = pd.read_csv("./train.csv", index_col="Id")

In [4]:
dataset = dataset.dropna(subset=["SalePrice"]).reset_index(drop=True)

X = dataset.iloc[:, :-1]
y = dataset.iloc[:, -1]

Rating colums with values:
* Ex (Excellent)
* Gd (Good)
* TA (Average/Typical)
* Fa (Fair)
* Po (Poor)
* NA

In [5]:
QUAL_COND_COLUMS = ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC", "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC"]
qual_cond_map = {"Ex" : 6, "Gd": 5, "TA" : 4, "Fa": 3, "Po" : 2, "NA" : 1}
X[QUAL_COND_COLUMS] = X[QUAL_COND_COLUMS].replace(qual_cond_map)

Utilities: Type of utilities available
* AllPub	All public Utilities (E,G,W,& S)	
* NoSewr	Electricity, Gas, and Water (Septic Tank)
* NoSeWa	Electricity and Gas Only
* ELO	Electricity only	

In [ ]:
util_map = {"AllPub": {"Electricity": 1, "Gas": 1, "Water": 1, "Septic": 1},
            "NoSewr": {"Electricity": 1, "Gas": 1, "Water": 1, "Septic": 0},
            "NoSeWa": {"Electricity": 1, "Gas": 1, "Water": 0, "Septic": 0},
            "ELO": {"Electricity": 1, "Gas": 0, "Water": 0, "Septic": 0},
            np.nan: {"Electricity": 0, "Gas": 0, "Water": 0, "Septic": 0},
            }

util_colums = pd.DataFrame(list(X["Utilities"].map(util_map)))
X = X.join(util_colums).drop(columns=["Utilities"])

In [7]:
CATEGORICAL_VALUES = ["MSZoning", "Street", "Alley", "LotShape", "LandContour", "LotConfig", "LandSlope", "Neighborhood", "Condition1", "Condition2", \
                      "BldgType", "HouseStyle", "RoofStyle", "RoofMatl", "Exterior1st", "Exterior2nd", "MasVnrType", "Foundation",  \
                      "BsmtExposure", "BsmtFinType1", "BsmtFinType2", "Heating", "CentralAir", "Electrical", "Functional", \
                      "GarageType", "GarageFinish", "PavedDrive", "Fence", "MiscFeature", "SaleType", "SaleCondition"]

In [8]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoded = encoder.fit_transform(X[CATEGORICAL_VALUES])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(CATEGORICAL_VALUES))

print(f"X shape: {X.shape}")
print(f"Encoded columns shape: {encoded_df.shape}")

X = pd.concat([X.drop(columns=CATEGORICAL_VALUES), encoded_df], axis=1)

print(f"New X shape: {X.shape}")

X shape: (1460, 82)
Encoded columns shape: (1460, 215)
New X shape: (1460, 265)


In [9]:
# X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

X_train = X
y_train = np.log1p(y)

In [10]:
imputer = SimpleImputer(strategy="most_frequent")
imputed = imputer.fit_transform(X_train)
X_train = pd.DataFrame(imputed, columns=imputer.get_feature_names_out())

In [11]:
NUMERICAL_VALUES = ["LotFrontage", "LotArea", "YearBuilt", "YearRemodAdd", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",\
                    "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea", "BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath",\
                    "TotRmsAbvGrd", "Fireplaces", "GarageYrBlt", "GarageCars", "GarageArea", "WoodDeckSF",\
                    "OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch", "PoolArea", "MiscVal", "MoSold", "YrSold", \
                    "ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC", "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC"]

In [12]:
scaler = StandardScaler()
X_train[NUMERICAL_VALUES] = scaler.fit_transform(X_train[NUMERICAL_VALUES])

In [126]:
X_train.shape

(1460, 263)

In [13]:
reg = RandomForestRegressor(random_state=42)

In [14]:
reg.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease o

In [23]:
test_data = pd.read_csv("./test.csv", index_col="Id")
X_test = test_data

list(X_test["Utilities"].map(util_map))

[{'Electricity': 1, 'Gas': 1, 'Water': 1, 'Septic': 1},
 {'Electricity': 1, 'Gas': 1, 'Water': 1, 'Septic': 1},
 {'Electricity': 1, 'Gas': 1, 'Water': 1, 'Septic': 1},
 {'Electricity': 1, 'Gas': 1, 'Water': 1, 'Septic': 1},
 {'Electricity': 1, 'Gas': 1, 'Water': 1, 'Septic': 1},
 {'Electricity': 1, 'Gas': 1, 'Water': 1, 'Septic': 1},
 {'Electricity': 1, 'Gas': 1, 'Water': 1, 'Septic': 1},
 {'Electricity': 1, 'Gas': 1, 'Water': 1, 'Septic': 1},
 {'Electricity': 1, 'Gas': 1, 'Water': 1, 'Septic': 1},
 {'Electricity': 1, 'Gas': 1, 'Water': 1, 'Septic': 1},
 {'Electricity': 1, 'Gas': 1, 'Water': 1, 'Septic': 1},
 {'Electricity': 1, 'Gas': 1, 'Water': 1, 'Septic': 1},
 {'Electricity': 1, 'Gas': 1, 'Water': 1, 'Septic': 1},
 {'Electricity': 1, 'Gas': 1, 'Water': 1, 'Septic': 1},
 {'Electricity': 1, 'Gas': 1, 'Water': 1, 'Septic': 1},
 {'Electricity': 1, 'Gas': 1, 'Water': 1, 'Septic': 1},
 {'Electricity': 1, 'Gas': 1, 'Water': 1, 'Septic': 1},
 {'Electricity': 1, 'Gas': 1, 'Water': 1, 'Septi

In [18]:
X_test[QUAL_COND_COLUMS] = X_test[QUAL_COND_COLUMS].replace(qual_cond_map)

util_colums = pd.DataFrame(list(X_test["Utilities"].map(util_map)))
X_test = X_test.join(util_colums).drop(columns=["Utilities"])

new_encoded = encoder.transform(X_test[CATEGORICAL_VALUES])
new_encoded_df = pd.DataFrame(new_encoded, columns=encoder.get_feature_names_out(CATEGORICAL_VALUES))

X_test = X_test.join(new_encoded_df).drop(columns=CATEGORICAL_VALUES)

new_encoded_df.shape, X_test.shape

new_imputed = imputer.transform(X_test)
X_test = pd.DataFrame(new_imputed, columns=imputer.get_feature_names_out())
X_test[NUMERICAL_VALUES] = scaler.transform(X_test[NUMERICAL_VALUES])

TypeError: 'float' object is not iterable

In [124]:
y_pred = reg.predict(X_test)

In [142]:
result = pd.concat([pd.DataFrame(X_test.index), pd.DataFrame(y_pred, columns=["SalePrice"])], axis=1)

In [145]:
result.to_csv("my_result.csv", index=False)